In [1]:
import torch
import math

def positional_encoding(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            pe[pos, i] = math.sin(pos / (10000 ** ((2 * i)/d_model)))
            if i + 1 < d_model:
                pe[pos, i + 1] = math.cos(pos / (10000 ** ((2 * i)/d_model)))
    return pe

# Example usage
print(positional_encoding(10, 16).shape)  # (10, 16)


torch.Size([10, 16])


In [3]:
positional_encoding(10,16)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00,  1.0000e-04,  1.0000e+00,
          1.0000e-05,  1.0000e+00,  1.0000e-06,  1.0000e+00,  1.0000e-07,
          1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00,  2.0000e-04,  1.0000e+00,
          2.0000e-05,  1.0000e+00,  2.0000e-06,  1.0000e+00,  2.0000e-07,
          1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9996e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00,  3.0000e-04,  1.0000e+00,
          3.0000e-05,  1.0000e+00,  3.0000e-06,  1.0000e+00,  3.0000e-07,
          1.0000e+00],
        [-7.5680e-01

In [10]:
import torch.nn as nn

class MiniTransformerBlock(nn.Module):
    def __init__(self, embed_dim, heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, 4*embed_dim),
            nn.ReLU(),
            nn.Linear(4*embed_dim, embed_dim)
        )
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_output)
        ff_output = self.ff(x)
        return self.norm2(x + ff_output)

    # Example usage:
    model = MiniTransformerBlock(embed_dim=16, heads=2)
    print(model)


MiniTransformerBlock(
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=16, out_features=16, bias=True)
  )
  (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (ff): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=16, bias=True)
  )
  (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
)


In [8]:
%pip install torchviz



Note: you may need to restart the kernel to use updated packages.


In [14]:
from torchviz import make_dot

dummy_input = torch.randn(1, 3, 224, 224)
output = model(dummy_input)

make_dot(output, params=dict(list(model.named_parameters()))).render("model")


'model.pdf'

In [ ]:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")

    input = tokenizer("Once upon a time", return_tensors="pt")
    output = model.generate(**input, max_length=30)
    print(tokenizer.decode(output[0]))


SSLError: (MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /gpt2/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))"), '(Request ID: edc1468a-0723-47ed-9978-4fa83b7ac42a)')